# TLMAC Integrated Flow

Full pipeline from N2UQ quantised ResNet-18 to FPGA LUT INIT files:

1. Load N2UQ quantised weights from `real_res18.pth.tar`
2. For each 3x3 conv layer: extract unique weight groups
3. Cluster steps that share weight groups (horizontal placement)
4. Simulated annealing for routing reduction (vertical placement)
5. Generate LUT-6 INIT truth tables
6. Export all TLMAC hex files

Reference: Gerlinghoff et al., FPGA 2024 | N2UQ: Liu et al., CVPR 2022

In [ ]:
!apt-get install -y git-lfs > /dev/null 2>&1 && git lfs install
!git clone https://github.com/leozqi/resnet18_n2uq.git || true
%cd resnet18_n2uq
!pip install torch torchvision numpy matplotlib 2>/dev/null

In [ ]:
import math, os, json
from collections import defaultdict
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt

torch.manual_seed(42)
np.random.seed(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. N2UQ Architecture (inline from resnet.py)

In [ ]:
class LearnableBias(nn.Module):
    def __init__(self, out_chn):
        super().__init__()
        self.bias = nn.Parameter(torch.zeros(1, out_chn, 1, 1))
    def forward(self, x):
        return x + self.bias.expand_as(x)

class LTQ(nn.Module):
    """Learnable Threshold Quantiser from N2UQ."""
    def __init__(self, num_bits):
        super().__init__()
        self.n_val = 2 ** num_bits - 1
        self.interval = 2.0 / self.n_val
        self.start = nn.Parameter(torch.tensor([0.0]))
        self.a = nn.Parameter(torch.tensor([self.interval] * self.n_val))
        self.scale1 = nn.Parameter(torch.tensor([1.0]))
        self.scale2 = nn.Parameter(torch.tensor([1.0]))

    def forward(self, x):
        x = x * self.scale1
        a_pos = torch.where(self.a > 1e-3, self.a, torch.tensor(1e-3))
        step_right = torch.tensor(0.0)
        x_f = x
        x_b = x
        for i in range(self.n_val):
            step_right = step_right + self.interval
            if i == 0:
                th_f = self.start + a_pos[0] / 2
                th_b = self.start
                x_f = torch.where(x > th_f, step_right, torch.tensor(0.0))
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, torch.tensor(0.0))
            else:
                th_f = th_f + a_pos[i - 1] / 2 + a_pos[i] / 2
                th_b = th_b + a_pos[i - 1]
                x_f = torch.where(x > th_f, step_right, x_f)
                x_b = torch.where(x > th_b, self.interval / a_pos[i] * (x - th_b) + step_right - self.interval, x_b)
        th_b = th_b + a_pos[-1]
        x_b = torch.where(x > th_b, torch.tensor(2.0), x_b)
        return (x_f.detach() + x_b - x_b.detach()) * self.scale2

class HardQuantizeConv(nn.Module):
    """Quantised convolution (HardQuantize) from N2UQ."""
    def __init__(self, in_chn, out_chn, num_bits, kernel_size=3, stride=1, padding=1):
        super().__init__()
        self.stride = stride
        self.padding = padding
        self.num_bits = num_bits
        self.kernel_size = kernel_size
        self.clip_val = nn.Parameter(torch.tensor([2.0]))
        self.shape = (out_chn, in_chn, kernel_size, kernel_size)
        self.weight = nn.Parameter((torch.rand(self.shape) - 0.5) * 0.001)

    def forward(self, x):
        real_weights = self.weight
        gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
        sf = gamma * real_weights.abs().mean(dim=[1, 2, 3], keepdim=True)
        sf = sf.detach()
        sw = real_weights / sf
        cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
        n = float(2 ** self.num_bits - 1) / self.clip_val
        qw = sf * (torch.round((cw + self.clip_val / 2) * n) / n - self.clip_val / 2)
        return F.conv2d(x, qw, stride=self.stride, padding=self.padding)

    def get_integer_weights(self):
        """Return quantised integer weights for TLMAC extraction."""
        with torch.no_grad():
            rw = self.weight
            gamma = (2**self.num_bits - 1) / (2**(self.num_bits - 1))
            sf = gamma * rw.abs().mean(dim=[1, 2, 3], keepdim=True)
            sw = rw / sf
            cw = sw.clamp(-self.clip_val / 2, self.clip_val / 2)
            n = float(2 ** self.num_bits - 1) / self.clip_val
            qi = torch.round((cw + self.clip_val / 2) * n)  # 0..2^n-1
        return qi.to(torch.int8)

In [ ]:
N_BIT = 3
QUANTIZE_DOWNSAMPLE = True

class BasicBlockQ(nn.Module):
    def __init__(self, inplanes, planes, stride=1, downsample=None):
        super().__init__()
        self.bias11 = LearnableBias(inplanes)
        self.prelu1 = nn.PReLU(inplanes)
        self.bias12 = LearnableBias(inplanes)
        self.quan1 = LTQ(N_BIT)
        self.conv1 = HardQuantizeConv(inplanes, planes, N_BIT, 3, stride, 1)
        self.bn1 = nn.BatchNorm2d(planes)
        self.bias21 = LearnableBias(planes)
        self.prelu2 = nn.PReLU(planes)
        self.bias22 = LearnableBias(planes)
        self.quan2 = LTQ(N_BIT)
        self.conv2 = HardQuantizeConv(planes, planes, N_BIT, 3, 1, 1)
        self.bn2 = nn.BatchNorm2d(planes)
        self.downsample = downsample
        self.bias31 = LearnableBias(planes)
        self.prelu3 = nn.PReLU(planes)
        self.bias32 = LearnableBias(planes)

    def forward(self, x):
        identity = x
        out = self.bias12(self.prelu1(self.bias11(x)))
        out = self.bn1(self.conv1(self.quan1(out)))
        out = self.bias22(self.prelu2(self.bias21(out)))
        out = self.bn2(self.conv2(self.quan2(out)))
        if self.downsample is not None:
            identity = self.downsample(x)
        out += identity
        out = self.bias32(self.prelu3(self.bias31(out)))
        return out

class N2UQ_ResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 64, 7, 2, 3, bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(3, 2, 1)
        self.layer1 = self._make_block(64, 64, 2)
        self.layer2 = self._make_block(64, 128, 2, stride=2)
        self.layer3 = self._make_block(128, 256, 2, stride=2)
        self.layer4 = self._make_block(256, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc = nn.Linear(512, 1000)

    def _make_block(self, inplanes, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or inplanes != planes:
            if QUANTIZE_DOWNSAMPLE:
                downsample = nn.Sequential(
                    LTQ(N_BIT),
                    HardQuantizeConv(inplanes, planes, N_BIT, 1, stride, 0),
                    nn.BatchNorm2d(planes))
            else:
                downsample = nn.Sequential(
                    nn.Conv2d(inplanes, planes, 1, stride, bias=False),
                    nn.BatchNorm2d(planes))
        layers = [BasicBlockQ(inplanes, planes, stride, downsample)]
        for _ in range(1, blocks):
            layers.append(BasicBlockQ(planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x = self.relu(self.bn1(self.conv1(x)))
        x = self.maxpool(x)
        x = self.layer4(self.layer3(self.layer2(self.layer1(x))))
        x = self.fc(torch.flatten(self.avgpool(x), 1))
        return x

model_n2uq = N2UQ_ResNet18().to(device)

CKPT = 'pretrained_models/quantize_downsampling_true/real_res18.pth.tar'
ckpt = torch.load(CKPT, map_location='cpu')
if isinstance(ckpt, dict) and 'state_dict' in ckpt:
    ckpt = ckpt['state_dict']
model_n2uq.load_state_dict(ckpt, strict=False)
model_n2uq.eval()
for p in model_n2uq.parameters(): p.requires_grad_(False)

print(f'N2UQ ResNet-18 loaded: {CKPT}')
print(f'Parameters: {sum(p.numel() for p in model_n2uq.parameters()):,}')

## 2. Identify 3x3 Conv Layers

In [ ]:
conv_layers = []
for name, module in model_n2uq.named_modules():
    if isinstance(module, HardQuantizeConv) and module.kernel_size in (3, (3, 3)):
        conv_layers.append((name, module))
        print(f'  {name}: {module.shape}  bits={module.num_bits}')
print(f'Total 3x3 conv layers: {len(conv_layers)}')


## 3. TLMAC Parameters & Helpers

G=3 (weight group = one kernel row), BW=3, NCLUS=8 select codes, LUT-6.

In [ ]:
G = 3  # weight group size
BW = N_BIT  # weight bit width
NCLUS = 2 ** (6 - G)  # 8
NLUT_PER_ARR = BW + math.ceil(math.log2(G))  # 5
DK = 3

print(f'TLMAC: G={G}, BW={BW}, NCLUS={NCLUS}, NLUT_PER_ARR={NLUT_PER_ARR}')

def extract_weight_groups(w_q):
    """w_q: [D_o, D_i, 3, 3] int weights (torch tensor on CPU).
    Returns wg_assign[D_s, D_p] as numpy, group_id_map, D_s, D_p."""
    D_o, D_i, _, _ = w_q.shape
    D_s = D_i * DK
    D_p = D_o
    w = w_q.reshape(D_o, D_s, DK).permute(1, 0, 2).reshape(-1, G)
    unique_groups, inverse = torch.unique(w, dim=0, return_inverse=True)
    wg = inverse.reshape(D_s, D_p).numpy()
    gid_map = {tuple(unique_groups[i].tolist()): i for i in range(len(unique_groups))}
    return wg, gid_map, D_s, D_p

def greedy_cluster_torch(C_bool, nclus):
    """Torch-vectorised agglomerative clustering.
    C_bool: [D_s, N_UWG] bool tensor on device.
    Returns list of step-set clusters, list of group-set per cluster.
    """
    DS, NG = C_bool.shape
    # Pack each row into a bitmask stored as int64 chunks.
    # For NG <= 64, one chunk; for NG <= 128, two chunks, etc.
    n_chunks = (NG + 63) // 64
    # mask_table[i] = bitmask for step i: [n_chunks] int64
    # Build by packing C_bool into int64 words
    C_int = C_bool.to(torch.int64)  # [D_s, N_UWG]
    # Pad to multiple of 64
    pad = n_chunks * 64 - NG
    if pad > 0:
        C_pad = F.pad(C_int, (0, pad))  # [D_s, n_chunks*64]
    else:
        C_pad = C_int
    # Reshape to [D_s, n_chunks, 64] and pack
    C_chunked = C_pad.reshape(DS, n_chunks, 64)
    bits = torch.arange(64, device=C_bool.device)
    step_masks = (C_chunked << bits).sum(dim=2)  # [D_s, n_chunks] int64

    # Cluster state: each cluster has a mask [n_chunks] and a set of step indices
    cluster_masks = step_masks.clone()  # [n_clusters, n_chunks]
    # Initially each step is its own cluster
    n_cl = DS

    # Popcount for int64 using the bit-twiddling approach (works on CPU)
    def popcount_tensor(t):
        """Popcount of int64 tensor. Returns sum of set bits."""
        t = t - ((t >> 1) & 0x5555555555555555)
        t = (t & 0x3333333333333333) + ((t >> 2) & 0x3333333333333333)
        t = (t + (t >> 4)) & 0x0F0F0F0F0F0F0F0F
        return (t * 0x0101010101010101) >> 56

    while n_cl > nclus:
        # Compute pairwise union sizes: [n_cl, n_cl]
        # cluster_masks: [n_cl, n_chunks]
        # union(i,j) = cluster_masks[i] | cluster_masks[j]
        # |union| = popcount(union)
        # To avoid n^2 * n_chunks memory, process in blocks
        
        best_i, best_j = 0, 1
        best_sz = DS + 1  # upper bound
        
        # Vectorised: compute OR of all pairs using broadcasting
        # [n_cl, 1, n_chunks] | [1, n_cl, n_chunks] -> [n_cl, n_cl, n_chunks]
        cm_i = cluster_masks.unsqueeze(1)  # [n_cl, 1, n_chunks]
        cm_j = cluster_masks.unsqueeze(0)  # [1, n_cl, n_chunks]
        
        # For large n_cl, do this in row-blocks to limit memory
        BLOCK = 128
        for row_start in range(0, n_cl, BLOCK):
            row_end = min(row_start + BLOCK, n_cl)
            block_i = cm_i[row_start:row_end]  # [bs, 1, n_chunks]
            union = block_i | cm_j  # [bs, n_cl, n_chunks]
            # Popcount across chunks
            pcs = popcount_tensor(union)  # [bs, n_cl, n_chunks]
            sizes = pcs.sum(dim=2)  # [bs, n_cl]
            # Zero out diagonal (self-merges) and lower triangle
            for ri in range(row_end - row_start):
                abs_i = row_start + ri
                sizes[ri, :abs_i + 1] = DS + 1  # mask out self and already-done
            # Find local minimum
            local_min = sizes.min()
            if local_min < best_sz:
                best_sz = local_min.item()
                flat_idx = sizes.argmin().item()
                best_i = row_start + flat_idx // n_cl
                best_j = flat_idx % n_cl
        
        # Merge j into i
        cluster_masks[best_i] = cluster_masks[best_i] | cluster_masks[best_j]
        # Remove j by replacing with last and shrinking
        if best_j < n_cl - 1:
            cluster_masks[best_j] = cluster_masks[n_cl - 1]
        n_cl -= 1
        # Track step membership: we need to map cluster index -> step sets
        # For efficiency, use a list parallel to cluster_masks
        # Rebuild clusters list from scratch after the loop

    # Reconstruct clusters: assign each step to its nearest cluster
    # by finding which cluster's mask contains all the step's groups
    final_masks = cluster_masks[:n_cl]
    # For each step, find the cluster whose mask is a superset
    # Use bitwise AND: if (step_mask & cluster_mask) == step_mask, step fits
    clusters = [set() for _ in range(n_cl)]
    for s in range(DS):
        sm = step_masks[s]  # [n_chunks]
        # Check which cluster contains this step
        # superset check: (sm & cm) == sm for all chunks
        overlap = (sm.unsqueeze(0) & final_masks) == sm.unsqueeze(0)  # [n_cl, n_chunks]
        fits = overlap.all(dim=1)  # [n_cl]
        cluster_idx = fits.nonzero(as_tuple=True)[0]
        if len(cluster_idx) > 0:
            clusters[cluster_idx[0].item()].add(s)
        else:
            # Fallback: assign to smallest cluster
            sizes = [(len(clusters[c]), c) for c in range(n_cl)]
            clusters[min(sizes)[1]].add(s)

    # Compute group sets per cluster
    cg = []
    for cl in clusters:
        gset = set()
        for s in cl:
            for g in range(NG):
                if C_bool[s, g]: gset.add(g)
        cg.append(gset)
    return clusters, cg

def routing_sa(clusters, wg_torch, group_to_outs, n_arr, dp):
    """SA for routing reduction. wg_torch: [D_s, D_p] int tensor."""
    ITERS = 10000
    ALPHA = 1.4
    cg_outs = []
    for cl in clusters:
        go = defaultdict(set)
        for s in cl:
            row = wg_torch[s].tolist()
            for p in range(dp):
                go[row[p]].add(p)
        cg_outs.append(dict(go))
    init_pp = [{gid: np.random.randint(0, n_arr) for gid in go} for go in cg_outs]

    def cnt(pp):
        used = np.zeros((n_arr, dp), dtype=bool)
        for assign in pp:
            for gid, arr in assign.items():
                for p in group_to_outs.get(gid, set()):
                    used[arr, p] = True
        return int(used.sum())

    curr = [dict(a) for a in init_pp]
    curr_r = cnt(curr)
    best = [dict(a) for a in curr]
    best_r = curr_r
    hist = []
    for it in range(1, ITERS + 1):
        T = ITERS / (it + 1) ** ALPHA
        c = np.random.randint(0, len(clusters))
        gids = list(curr[c].keys())
        if len(gids) < 2:
            hist.append((it, curr_r, best_r))
            continue
        i0, i1 = np.random.choice(len(gids), 2, replace=False)
        nn_pp = [dict(a) for a in curr]
        nn_pp[c][gids[i0]], nn_pp[c][gids[i1]] = nn_pp[c][gids[i1]], nn_pp[c][gids[i0]]
        nr = cnt(nn_pp)
        if nr < curr_r or np.random.random() < math.exp(-(nr - curr_r + 1) / max(T, 1e-12)):
            curr, curr_r = nn_pp, nr
            if curr_r < best_r:
                best_r, best = curr_r, [dict(a) for a in curr]
        hist.append((it, curr_r, best_r))
    return best, best_r, hist


## 4. Compile All Layers

In [ ]:
all_results = {}
wg_assign_all = {}  # layer_name -> wg_assign array (numpy)

for name, module in conv_layers:
    wq = module.get_integer_weights()  # [D_o, D_i, 3, 3] int8 tensor
    wg, gid_map, D_s, D_p = extract_weight_groups(wq)
    N_UWG = len(gid_map)
    wg_assign_all[name] = wg

    # Build C as torch bool tensor [D_s, N_UWG]
    wg_flat = torch.from_numpy(wg).to(torch.int64)
    C = torch.zeros(D_s, N_UWG, dtype=torch.bool)
    # scatter: for each (s, p), C[s, wg[s,p]] = True
    # Use advanced indexing
    s_idx = torch.arange(D_s).unsqueeze(1).expand(-1, D_p).reshape(-1)
    g_idx = wg_flat.reshape(-1)
    C[s_idx, g_idx] = True

    clusters, cg = greedy_cluster_torch(C, NCLUS)
    n_arr = max(len(g) for g in cg)

    # Group-to-outputs mapping (torch scatter)
    g2o = defaultdict(set)
    for s in range(D_s):
        row = wg_flat[s].tolist()
        for p in range(D_p):
            g2o[row[p]].add(p)
    g2o = dict(g2o)

    # Routing SA (use torch tensor for wg assignment)
    best_pp, best_r, hist = routing_sa(clusters, wg_flat, g2o, n_arr, D_p)

    # Step-to-cluster mapping
    stc = {}
    for ci, cl in enumerate(clusters):
        for s in cl:
            stc[s] = ci

    # LUT INIT generation (vectorised per array)
    id2g = {v: k for k, v in gid_map.items()}
    zero_gid = gid_map.get((0,) * G, 0)
    addr_range = torch.arange(64)
    sel_all = (addr_range >> G) & (NCLUS - 1)
    abits_all = addr_range & ((1 << G) - 1)
    # Precompute bit extraction for all 3 activations
    act_bits_tensor = torch.stack([(abits_all >> gi) & 1 for gi in range(G)], dim=1)  # [64, G]

    all_init = {}
    for ci in range(len(clusters)):
        for ai in range(n_arr):
            # Map select codes to group IDs at this array
            sg = {}
            for gid, arr in best_pp[ci].items():
                if arr == ai:
                    sg[len(sg)] = gid
            sg_full = [sg.get(sel, zero_gid) for sel in range(NCLUS)]
            # Build weight matrix: [NCLUS, G] for this array
            wg_mat = torch.zeros(NCLUS, G, dtype=torch.int64)
            for sel in range(NCLUS):
                gid = sg_full[sel]
                w = id2g.get(gid, (0,) * G)
                for gi in range(G):
                    wg_mat[sel, gi] = w[gi]
            # Compute MAC: [64] = sum over G of act_bits * wg
            sel_idx = sel_all.long()
            sel_wg = wg_mat[sel_idx]  # [64, G]
            mac = (act_bits_tensor * sel_wg).sum(dim=1)  # [64]
            # Handle negative values for 2's complement
            mac_np = mac.numpy()
            mac_np = np.where(mac_np < 0, mac_np + (1 << NLUT_PER_ARR), mac_np)
            # Pack into INIT values
            iv = [0] * NLUT_PER_ARR
            for addr in range(64):
                val = int(mac_np[addr])
                for k in range(NLUT_PER_ARR):
                    iv[k] |= ((val >> k) & 1) << addr
            all_init[(ci, ai)] = iv

    all_results[name] = dict(
        D_s=D_s, D_p=D_p, N_UWG=N_UWG,
        n_clusters=len(clusters), n_arr=n_arr,
        best_routes=best_r, possible=n_arr * D_p,
        all_init=all_init, step_to_cluster=stc,
        best_placement=best_pp, history=hist,
        id_to_group=id2g, gid_map=gid_map,
    )
    route_pct = 1.0 - best_r / (n_arr * D_p) if n_arr * D_p > 0 else 0.0
    short = name.replace('.conv', '.c')
    print(f'{short}: {N_UWG} uwg, {len(clusters)} cl, {n_arr} arr, '
          f'routes {best_r}/{n_arr*D_p} ({route_pct:.0%} red)')

print(f'Compiled {len(all_results)} layers')


## 5. Routing SA Convergence

In [ ]:
n_layers = len(all_results)
if n_layers == 0:
    print('No layers compiled — check previous cell for errors.')
else:
    ncols = min(5, n_layers)
    nrows = (n_layers + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3 * nrows))
    if n_layers == 1:
        axes = np.array([axes])
    axes_flat = np.array(axes).flatten() if hasattr(axes, 'flatten') else [axes]
    for idx, (name, res) in enumerate(all_results.items()):
        ax = axes_flat[idx]
        iters = [h[0] for h in res['history']]
        curr  = [h[1] for h in res['history']]
        best  = [h[2] for h in res['history']]
        ax.plot(iters, curr, alpha=0.3, linewidth=0.5)
        ax.plot(iters, best, linewidth=1.5)
        short = name.replace('.conv', '.c')
        ax.set_title(f"{short}\n{res['N_UWG']} grps, {res['n_arr']} arr", fontsize=8)
        ax.tick_params(labelsize=7)
    for idx in range(n_layers, len(axes_flat)):
        axes_flat[idx].set_visible(False)
    plt.tight_layout()
    plt.show()


In [ ]:
total_lut6 = 0
print(f"{'Layer':<30s} {'UWG':>5s} {'Cl':>4s} {'N_arr':>5s} {'LUT-6':>7s} {'Routes':>10s}")
print('-' * 65)
for name, res in all_results.items():
    lut6 = res['n_arr'] * NLUT_PER_ARR * res['n_clusters']
    total_lut6 += lut6
    short = name.replace('.conv', '.c')
    print(f"{short:<30s} {res['N_UWG']:>5d} {res['n_clusters']:>4d} {res['n_arr']:>5d} {lut6:>7d} {res['best_routes']:>10d}")
print('-' * 65)
print(f'Total LUT-6: {total_lut6:,}')

## 6. Export TLMAC Hex Files

Writes to `tlmac_output/weights/{layer}/`.
Files match the `INIT_FILE` parameter of `lut_array.sv` and `cluster_map_rom.sv`.

In [ ]:
OUT_DIR = 'tlmac_output'
os.makedirs(os.path.join(OUT_DIR, 'weights'), exist_ok=True)
metadata = {}

for name, res in all_results.items():
    safe = name.replace('.', '_')
    ldir = os.path.join(OUT_DIR, 'weights', safe)
    os.makedirs(ldir, exist_ok=True)
    D_s = res['D_s']
    n_arr = res['n_arr']

    # cluster_map.hex: one line per step
    with open(os.path.join(ldir, 'cluster_map.hex'), 'w') as f:
        for s in range(D_s):
            f.write(f"{res['step_to_cluster'].get(s, 0):02X}\n")

    # lut_arrN.hex: NLUT_PER_ARR lines per cluster, one file per array
    for ai in range(n_arr):
        with open(os.path.join(ldir, f'lut_arr{ai}.hex'), 'w') as f:
            for ci in range(res['n_clusters']):
                for val in res['all_init'][(ci, ai)]:
                    f.write(f"{val:016X}\n")

    # routing_bitmap.hex
    wg = wg_assign_all[name]
    g2o = defaultdict(set)
    for s in range(D_s):
        for p in range(res['D_p']):
            g2o[wg[s, p]].add(p)
    R = np.zeros((n_arr, res['D_p']), dtype=bool)
    for ci, assign in enumerate(res['best_placement']):
        for gid, ai in assign.items():
            for p in g2o.get(gid, set()):
                R[ai, p] = True
    with open(os.path.join(ldir, 'routing_bitmap.hex'), 'w') as f:
        for e in range(n_arr):
            bitmap = 0
            for p in range(res['D_p']):
                if R[e, p]: bitmap |= (1 << p)
            nw = (res['D_p'] + 63) // 64
            for wi in range(nw):
                f.write(f"{(bitmap >> (wi * 64)) & 0xFFFFFFFFFFFFFFFF:016X}\n")

    metadata[safe] = {
        'layer': name, 'D_s': D_s, 'D_p': res['D_p'],
        'N_UWG': res['N_UWG'], 'clusters': res['n_clusters'],
        'n_arr': n_arr, 'G': G, 'BW': BW, 'NLUT_PER_ARR': NLUT_PER_ARR, 'NCLUS': NCLUS,
        'best_routes': res['best_routes'], 'possible': res['possible'],
    }

with open(os.path.join(OUT_DIR, 'metadata.json'), 'w') as f:
    json.dump(metadata, f, indent=2)

print(f'Exported {len(metadata)} layers to {OUT_DIR}/')
for k in metadata:
    v = metadata[k]
    print(f"  {k}: {v['n_arr']} arr, {v['clusters']} cl, {v['best_routes']} routes")

# List output files
for root, dirs, files in os.walk(os.path.join(OUT_DIR, 'weights')):
    for fname in sorted(files):
        sz = os.path.getsize(os.path.join(root, fname))
        rel = os.path.relpath(os.path.join(root, fname), OUT_DIR)
        print(f'  {rel}  ({sz:,} bytes)')

In [ ]:
print('\n=== TLMAC Integrated Compilation Summary ===')
print(f'Model: N2UQ ResNet-18 ({N_BIT}-bit, quantize_downsample={QUANTIZE_DOWNSAMPLE})')
print(f'Layers compiled:       {len(all_results)}')
total_groups = sum(r['N_UWG'] for r in all_results.values())
print(f'Total unique groups:   {total_groups:,}')
total_arr = sum(r['n_arr'] for r in all_results.values())
print(f'Total LUT arrays:      {total_arr}')
print(f'Total LUT-6:           {total_lut6:,}')
print(f'Output:                {OUT_DIR}/')